# 02 · Três métodos de seleção e votação das features

Regra fixada **antes** da comparação (`config.yaml → votacao`): cada método ranqueia as features candidatas
e vota nas suas **k = 10** melhores. Uma feature é eleita com **≥ 2 dos 3 votos**. Se sobrarem menos de **6**,
completa-se pela menor média de ranking, que também desempata e ordena.

Os três métodos são ajustados **só com o treino do ciclo**, numa amostra estratificada de 200 mil linhas:

| Método | O que mede |
|---|---|
| `SelectKBest(mutual_info_classif)` | dependência estatística (inclusive não linear) entre cada feature e o alvo |
| Importância de `RandomForestClassifier` | quanto cada feature reduz a impureza nas árvores |
| `RFE(LogisticRegression)` | ordem em que a feature é eliminada por ter o menor coeficiente |

In [1]:
import sys, json, pathlib, warnings
RAIZ = pathlib.Path.cwd().resolve()
while not (RAIZ / "config" / "config.yaml").exists():  # funciona rodando de notebooks/ ou da raiz
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", message=".*Hint: Inferred schema.*")

import pandas as pd
from src.comum import carregar_config, caminho, estado, ler_ciclo, atualizar_ciclo, configurar_mlflow, tags_ciclo, pasta_evidencias, gatilho
cfg = carregar_config()
pd.set_option("display.width", 160)

In [2]:
from src import features as fs, preprocessamento as pp
ciclo = ler_ciclo(cfg)
treino = pd.read_parquet(estado(cfg, "processados") / "treino.parquet")
v = cfg["votacao"]
amostra = fs.amostrar(treino, v["amostra"], v["seed"])
print(ciclo["cycle_id"], ciclo["data_version"], "| amostra:", len(amostra), "linhas,",
      f"{amostra[pp.ALVO].mean():.2%} positivos | candidatas:", len(pp.CANDIDATAS))

20261124-0010-45294862870 vbase-3f4e3604 | amostra: 200000 linhas, 2.10% positivos | candidatas: 20


In [3]:
tabelas = {nome: metodo(amostra, pp.CANDIDATAS, v["k"], v["seed"]) for nome, metodo in fs.METODOS.items()}
pd.concat({n: t[["score", "ranking", "voto"]] for n, t in tabelas.items()}, axis=1).sort_values(("random_forest", "ranking")).round(4)

/opt/hostedtoolcache/Python/3.12.16/x64/lib/python3.12/site-packages/sklearn/preprocessing/_target_encoder.py:342: FutureWarning: `TargetEncoder.shuffle` and `TargetEncoder.random_state` are deprecated in version 1.9 and will be removed in version 1.11. Pass a cross-validation generator as `cv` argument to specify the shuffling behaviour instead.
  warnings.warn(


selectkbest_mi              random_forest              rfe_logistica             
                                score ranking voto         score ranking voto         score ranking voto
feature                                                                                                 
taxa_historica_periodo         0.0159       3    1        0.2434       1    1            20       1    1
taxa_historica                 0.0152       6    1        0.1900       2    1            10      11    0
acid_ult_365d                  0.0157       4    1        0.1454       3    1            17       4    1
vmd                            0.0161       2    1        0.0961       4    1             1      20    0
lon                            0.0164       1    1        0.0629       5    1            12       9    1
delegacia                      0.0075       7    1        0.0466       6    1            18       3    1
temperatura                    0.0007      13    0        0.0330       7    1             9      12    0
nebulosidade                   0.0007      12    0        0.0263       8    1             3      18    0
br                             0.0064       8    1        0.0260       9    1            19       2    1
trecho_km                      0.0035       9    1        0.0229      10    1            13       8    1
lat                            0.0156       5    1        0.0214      11    0            14       7    1
chuva_mm                       0.0009      10    1        0.0205      12    0            11      10    1
periodo_idx                    0.0008      11    0        0.0194      13    0            16       5    1
dia_semana                     0.0002      14    0        0.0175      14    0            15       6    1
mes                            0.0000      17    0        0.0144      15    0             4      17    0
fim_de_semana                  0.0001      15    0        0.0062      16    0             8      13    0
pandemia                       0.0001      16    0        0.0041      17    0             7      14    0
temporada_verao                0.0000      19    0        0.0023      18    0             5      16    0
vespera_ou_pos_feriado         0.0000      20    0        0.0010      19    0             2      19    0
feriado                        0.0000      18    0        0.0006      20    0             6      15    0

## Tabela de votos e features eleitas

In [4]:
votacao = fs.votar(tabelas, v["min_votos"], v["min_features"])
eleitas = list(votacao.index[votacao["eleita"]])
print(f"{len(eleitas)} features eleitas:", eleitas)
votacao

10 features eleitas: ['taxa_historica_periodo', 'acid_ult_365d', 'lon', 'delegacia', 'br', 'trecho_km', 'taxa_historica', 'lat', 'vmd', 'chuva_mm']


,voto_selectkbest_mi,voto_random_forest,voto_rfe_logistica,rank_selectkbest_mi,rank_random_forest,rank_rfe_logistica,votos,media_ranking,eleita,motivo
feature,,,,,,,,,,
taxa_historica_periodo,1,1,1,3,1,1,3,1.67,True,≥ 2 votos
acid_ult_365d,1,1,1,4,3,4,3,3.67,True,≥ 2 votos
lon,1,1,1,1,5,9,3,5.00,True,≥ 2 votos
delegacia,1,1,1,7,6,3,3,5.33,True,≥ 2 votos
br,1,1,1,8,9,2,3,6.33,True,≥ 2 votos
trecho_km,1,1,1,9,10,8,3,9.00,True,≥ 2 votos
taxa_historica,1,1,0,6,2,11,2,6.33,True,≥ 2 votos
lat,1,0,1,5,11,7,2,7.67,True,≥ 2 votos
vmd,1,1,0,2,4,20,2,8.67,True,≥ 2 votos


In [5]:
ev = pasta_evidencias(cfg, ciclo)
votacao.to_csv(ev / "votacao.csv")
mlflow = configurar_mlflow(cfg)
with mlflow.start_run(run_name=f"{ciclo['cycle_id']} · votação", tags=tags_ciclo(ciclo, etapa="votacao")) as run:
    mlflow.log_params({"metodos": ",".join(fs.METODOS), "k": v["k"], "min_votos": v["min_votos"],
                       "min_features": v["min_features"], "desempate": v["desempate"], "amostra": len(amostra),
                       "seed": v["seed"], "features_eleitas": ",".join(eleitas)})
    mlflow.log_metric("n_features_eleitas", len(eleitas))
    mlflow.log_artifact(ev / "votacao.csv")
    for nome, t in tabelas.items():
        t.to_csv(ev / f"ranking_{nome}.csv"); mlflow.log_artifact(ev / f"ranking_{nome}.csv", "rankings")
ciclo = atualizar_ciclo(cfg, features_eleitas=eleitas, run_votacao=run.info.run_id)